# Searching for Composition Matches Between Mini-Neptunes and Modern/Archean Earth: Optimizing the Photochem/PT Grid Part 1

In [1]:
import sys
from pathlib import Path

_root = Path.cwd()  # assumes notebook is run from the project root
sys.path.insert(0, str(_root / 'MiniNepGrid_Scripts'))
sys.path.insert(0, str(_root / 'ReflectedSpectra_Scripts'))

import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import h5py


current_directory = Path.cwd()
references_directory_path = "Installation&Setup_Instructions/picasofiles/reference"
PYSYN_directory_path = "Installation&Setup_Instructions/picasofiles/grp/redcat/trds"
print(os.path.join(current_directory, references_directory_path))
print(os.path.join(current_directory, PYSYN_directory_path))

os.environ['picaso_refdata']= os.path.join(current_directory, references_directory_path)
os.environ['PYSYN_CDBS']= os.path.join(current_directory, PYSYN_directory_path)

import picaso.justdoit as jdi
import picaso.justplotit as jpi

import copy
import pandas as pd

import numpy as np
from scipy import optimize
from matplotlib import pyplot as plt

import pickle
import astropy.units as u

import Photochem_grid_121625 as photochem_grid
import PICASO_Climate_grid_121625 as picaso_grid
import GraphsKey as GraphsKey

/mnt/c/Users/lily/Documents/NASAUWPostbac/MiniNeptuneGrid26_PostBac/Installation&Setup_Instructions/picasofiles/reference
/mnt/c/Users/lily/Documents/NASAUWPostbac/MiniNeptuneGrid26_PostBac/Installation&Setup_Instructions/picasofiles/grp/redcat/trds


/mnt/c/Users/lily/Documents/NASAUWPostbac/MiniNeptuneGrid26_PostBac/Installation&Setup_Instructions/picasofiles/reference
/mnt/c/Users/lily/Documents/NASAUWPostbac/MiniNeptuneGrid26_PostBac/Installation&Setup_Instructions/picasofiles/grp/redcat/trds


## Load Photochem Cases that converged

In [2]:
filename_photochem ='data/grid_results/Photochem_1D_updatop_paramext_reducedrad_full_try3.h5'
dictionary_inputs = []

with h5py.File(filename_photochem, 'r') as f:
    photochem_data = f
    print(photochem_data.keys())
    print(photochem_data['results'].keys())
    #print(list(photochem_data['inputs']))
    #print(list(photochem_data['completed']))
    #print(list(photochem_data['results']['error']))
    #print(photochem_data['results'].keys())

    pressure = f['results']['pressure_sol']
    temperature = f['results']['temperature_sol']
    converged_PC = f['results']['converged_PC']
    converged_TP = f['results']['converged_TP']
    error = f['results']['error']
    status = f['results']['status']

    # Get grid sizes
    Nr, Nm, Nt, Na, Nc, Nk = pressure.shape[:6]

    print(Nr, Nm, Nt, Na, Nc, Nk)

    for i in range(Nr):
        for j in range(Nm):
            for k in range(Nt):
                for l in range(Na):
                    for m in range(Nc):
                        for n in range(Nk):
                            print(i, j, k, l, m, n)
                            pressure_values = np.array(pressure[i, j, k, l, m, n])
                            temperature_values = np.array(temperature[i, j, k, l, m, n])
                            conv_value = np.array(converged_PC[i, j, k, l, m, n])
                            error_value = np.array(error[i, j, k, l, m, n]) 
                            status_value = np.array(status[i, j, k, l, m, n]) 
                            conv_TP_value = np.array(converged_TP[i, j, k, l, m, n])
                            #print((conv_value))
    
                            dictionary_inputs.append([i, j, k, l, m, n, conv_value, conv_TP_value, error_value, status_value])
                            #print(i, j, k, l, m, k, conv_value)
                            
                            arr = conv_value.copy()
                            zero_indices = np.where(arr == 0)[0]
    
                            #if zero_indices != 0:
                            #    print(f'Indices that did not converge: {zero_indices}')
                            
                            # Do something with them
                            #for value in conv_value:
                            #    if value == 0:
                            #        not_converged += 1
                            #        print(not_converged)

print(len(dictionary_inputs))

<KeysViewHDF5 ['completed', 'inputs', 'results']>
<KeysViewHDF5 ['1CH2_sol', '1CH2_soleq', 'C2H2OH_sol', 'C2H2OH_soleq', 'C2H2_sol', 'C2H2_soleq', 'C2H2aer_sol', 'C2H2aer_soleq', 'C2H3OH_sol', 'C2H3OH_soleq', 'C2H3_sol', 'C2H3_soleq', 'C2H4OH_sol', 'C2H4OH_soleq', 'C2H4_sol', 'C2H4_soleq', 'C2H4aer_sol', 'C2H4aer_soleq', 'C2H5_sol', 'C2H5_soleq', 'C2H6_sol', 'C2H6_soleq', 'C2H6aer_sol', 'C2H6aer_soleq', 'C2H_sol', 'C2H_soleq', 'C2_sol', 'C2_soleq', 'C3H4_sol', 'C3H4_soleq', 'C3H6_sol', 'C3H6_soleq', 'C4H2_sol', 'C4H2_soleq', 'C4H3_sol', 'C4H3_soleq', 'C4H4_sol', 'C4H4_soleq', 'C4H_sol', 'C4H_soleq', 'CH2CHO_sol', 'CH2CHO_soleq', 'CH2CN_sol', 'CH2CN_soleq', 'CH2CO_sol', 'CH2CO_soleq', 'CH2N2_sol', 'CH2N2_soleq', 'CH2_sol', 'CH2_soleq', 'CH3CHO_sol', 'CH3CHO_soleq', 'CH3CN_sol', 'CH3CN_soleq', 'CH3CNaer_sol', 'CH3CNaer_soleq', 'CH3CO_sol', 'CH3CO_soleq', 'CH3O2_sol', 'CH3O2_soleq', 'CH3OH_sol', 'CH3OH_soleq', 'CH3O_sol', 'CH3O_soleq', 'CH3_sol', 'CH3_soleq', 'CH4_sol', 'CH4_soleq', 'CH4a

In [3]:
converged_inputs = []
not_converged_inputs = []
not_converged_status = []
not_converged_status_unknown = []
photochem_error = []
picaso_error = []
picaso_not_converged = []
photochem_BOA_error = []
phtoochem_AE_error = []
phtoochem_EE_error = []

for inputs in dictionary_inputs:
    if inputs[6][0] == True:
        converged_inputs.append(inputs)

print(len(converged_inputs)) 
# Out of 7840 cases

for inputs in dictionary_inputs:
    if inputs[6][0] == False:
        not_converged_inputs.append(inputs)
        
print(len(not_converged_inputs))
print(not_converged_inputs[0][9])

count = 0
count2 = 0
count1 = 0
count3 = 0
count4 = 0

for inputs in not_converged_inputs:
    if inputs[9] == [b'Photochem-not-converged']:
        count += 1
        not_converged_status.append(inputs)

    elif inputs[9] == [b'Photochem-error']:
        count1 += 1
        photochem_error.append(inputs)

    elif inputs[9] == [b'PICASO: error']:
        count2 += 1
        picaso_error.append(inputs)

    elif inputs[9] == [b'PICASO: not_conve']:
        count3 += 1
        picaso_not_converged.append(inputs)

    else:
        count3 += 1
        not_converged_status_unknown.append(inputs)

count5 = 0
count6 = 0
count7 = 0

for inputs in photochem_error:
    if inputs[8] == [b'Exception: BOA in photochemical model wants to be deeper than BOA of climate model.']:
        photochem_BOA_error.append(inputs)
        count5 += 1

    elif inputs[8] == [b'AssertionError: ']:
        phtoochem_AE_error.append(inputs)
        count6 += 1

    elif inputs[8] == [b'EquilibrateException: ERROR: singular matrix in LUDCMP']:
        phtoochem_EE_error.append(inputs)
        count7 += 1

9088
2792
[b'Photochem-not-converged']


In [4]:
# Full Parameter Exploration
rad_plan_earth_units = np.array([2]) # in units of xEarth radii
log10_planet_metallicity = np.linspace(0.5, 3.5, 9) # in units of solar metallicity
print(log10_planet_metallicity)
tint_K = np.linspace(50, 400, 8) # in Kelvin
print(tint_K)
semi_major_AU = np.array([0.3, 0.7, 1, 1.5, 2, 3, 4, 5, 6, 8, 10]) # in AU 
ctoO_solar = np.linspace(0.01, 1, 5) # in units of solar C/O
print(ctoO_solar)
kzz_list = np.array([5, 7, 9])

[0.5   0.875 1.25  1.625 2.    2.375 2.75  3.125 3.5  ]
[ 50. 100. 150. 200. 250. 300. 350. 400.]
[0.01   0.2575 0.505  0.7525 1.    ]


In [5]:
converged_inputs_listed = []

for inputs in converged_inputs:

    rad = rad_plan_earth_units[inputs[0]]
    metal = log10_planet_metallicity[inputs[1]]
    tint = tint_K[inputs[2]]
    semi_major = semi_major_AU[inputs[3]]
    ctoO = ctoO_solar[inputs[4]]
    kzz = kzz_list[inputs[5]]

    inputs = [rad, metal, tint, semi_major, ctoO, kzz]
    converged_inputs_listed.append(inputs)
    #print(rad, metal, tint, semi_major, ctoO, kzz)

## Search for cases with similar compositions to Archean & Modern Earth
- Specifically where both Archean and/or Modern Earth and the Photochem Mini-Neptune have the same top molecules as each other (+ limited molecules I was checking for and limited pressure of the atmosphere); I do not have all cases I explored saved but (as I altered the get_top_molecules_for_case function manually for each of the following cases.

The case I ended up going with was the following Final Case:
- archean_required = {"N2", "CO2", "CO", "H2O", "CH4"}; ?? cases had these molecules in the top 12 molecules of 0 - 10 bars
- modern_required = {"N2", "O2", "O3", "H2O", "CO2"}; 33 cases had these molecules in the top 12 molecules of 0 - 10 bars (if I really want to I can decrease molecules to 10 from this sample size)

##### Other Cases Explored

In [6]:
# This is out of a sample of 100 cases, instead of the ~9000.

# archean_required = {"N2", "CO2", "CO", "H2O"}; 77 cases had these molecules in the top 12 molecules of full pressure spectrum
# modern_required = {"N2", "O2", "O3", "H2O", "CO2"}; 0 cases had these molecules in the top 12 molecules of full pressure spectrum 

#Note that some cases are matches for both archean and modern earth: Final Case???
# archean_required = {"N2", "CO2", "CO", "H2O"}; 52 cases had these molecules in the top 12 molecules of 0 - 10 bars
# modern_required = {"N2", "O2", "O3", "H2O", "CO2"}; 33 cases had these molecules in the top 12 molecules of 0 - 10 bars

#Note that some cases are matches for both archean and modern earth:
# archean_required = {"N2", "CO2", "CO", "H2O"}; 34 cases had these molecules in the top 10 molecules of 0 - 10 bars
# modern_required = {"N2", "O2", "O3", "H2O", "CO2"}; 8 cases had these molecules in the top 10 molecules of 0 - 10 bars

# archean_required = {"N2", "CO2", "CO", "H2O", "CH4"}; 16 cases had these molecules in the top 12 molecules of 0 - 10 bars
# modern_required = {"N2", "O2", "O3", "H2O", "CO2", "CH4"}; 0 cases had these molecules in the top 12 molecules of 0 - 10 bars

# Final Case??? Imma just go with the 52 & 33 one and separate them into top 10 molecules & seperate them into cases where CH4 is in the top 10 molecules
# archean_required = {"N2", "CO2", "CO", "H2O", "CH4"}; ?? cases had these molecules in the top 12 molecules of 0 - 10 bars
# modern_required = {"N2", "O2", "O3", "H2O", "CO2"}; 33 cases had these molecules in the top 12 molecules of 0 - 10 bars (if I really want to I can decrease molecules to 10 from this sample size)

In [7]:
df_mol_archean_earth = {
    "N2":0.945,
    "CO2":0.05,
    "CO":0.0005,
    "CH4":0.005, 
    "H2O":0.003
}

df_mol_modern_earth = {
    "N2":0.79,
    "O2":0.21,
    "O3":7e-7,
    "H2O":3e-3,
    "CO2":300e-6,
    "CH4":1.7e-6
}

In [8]:
def get_top_molecules_for_case(photochem, rad, metal, tint, sma, ctoO, kzz, top_n=15):
    excluded_suffixes = ('aer_sol', 'aer_soleq', 'soleq')
    excluded_exact = {
        'pressure_sol', 'temperature_sol', 'Kzz_sol',
        'pressure_soleq', 'temperature_soleq', 'Kzz_soleq',
        'converged_PC', 'converged_TP'
    }

    molecule_means = {}

    # --- Load pressure profile for this case ---
    pressure = photochem["pressure_sol"][rad][metal][tint][sma][ctoO][kzz]

    # Pressure is high → low, so select where 0 < P < 10 bar
    valid = (pressure > 0) & (pressure < 10)

    # If no valid region, skip this case
    if not np.any(valid):
        return [], {}

    for key in photochem.keys():
        if key in excluded_exact:
            continue
        if key.endswith(excluded_suffixes):
            continue

        # Try to extract dataset
        try:
            dataset = photochem[key][rad][metal][tint][sma][ctoO][kzz]
        except Exception:
            continue

        # Skip non-numeric datasets
        if not np.issubdtype(dataset.dtype, np.number):
            continue

        # Slice dataset using the pressure mask
        sliced = dataset[valid]

        # Compute mean mixing ratio in the 0–10 bar region
        mean_val = np.nanmean(sliced)
        molecule_means[key] = mean_val

    # Get top N molecules
    top_keys = heapq.nlargest(top_n, molecule_means, key=molecule_means.get)

    return top_keys, molecule_means


### The following cell runs the calculation for Modern and Archean cases finding mini-Neptune photochemistry with similar molecules in the top 12 above 10 bars. 

In [9]:
archean_required = {"N2", "CO2", "CO", "H2O"} #Try without CH4
modern_required  = {"N2", "O2", "O3", "H2O", "CO2"} #Try without CH4

archean_like_mini_neptunes = {}
modern_like_mini_neptunes = {}

filename_photochem ='data/grid_results/Photochem_1D_updatop_paramext_reducedrad_full_try3.h5'

test_inputs = converged_inputs
total = len(test_inputs)

with h5py.File(filename_photochem, 'r') as f:
    photochem = f['results']

    print(f"Beginning scan of {total} cases...")

    for i, inputs in enumerate(test_inputs):
        rad, metal, tint, sma, ctoO, kzz = inputs[:6]
        index_tuple = (rad, metal, tint, sma, ctoO, kzz)

        if i % 1 == 0:
            print(f"Processing case {i}/{total} (indices={index_tuple})")

        # Get top 10 molecules for this case
        top10, means = get_top_molecules_for_case(
            photochem, rad, metal, tint, sma, ctoO, kzz, top_n=12
        )

        top10_clean = {mol.replace("_sol", "") for mol in top10}

        # Check Archean
        if archean_required.issubset(top10_clean):
            archean_like_mini_neptunes[index_tuple] = {
                "indices": index_tuple,
                "top10": top10_clean
            }
            print(f'Archean match found')

        # Check Modern
        if modern_required.issubset(top10_clean):
            modern_like_mini_neptunes[index_tuple] = {
                "indices": index_tuple,
                "top10": top10_clean
            }
            print(f'Modern match found')

    print("Scan complete.")
    print(f"Archean-like cases found: {len(archean_like_mini_neptunes)}")
    print(f"Modern-like cases found: {len(modern_like_mini_neptunes)}")


Beginning scan of 9088 cases...
Processing case 0/9088 (indices=(0, 0, 0, 0, 0, 0))



KeyboardInterrupt



### Saves the files or opens one that has been created already.

In [147]:
import pickle

#with open("GridAnalysisSteps1_2_files/mini_neptune_classification.pkl", "wb") as f:
#    pickle.dump(
#        {
#            "archean": archean_like_mini_neptunes,
#            "modern": modern_like_mini_neptunes
#        },
#        f
#    )

In [10]:
with open("GridAnalysisSteps1_2_files/mini_neptune_classification.pkl", "rb") as f:
    data = pickle.load(f)

archean_like_mini_neptunes = data["archean"]
modern_like_mini_neptunes = data["modern"]

print(f'This was how many Mini-Neptunes matched Archean Earth: {len(archean_like_mini_neptunes.keys())}')
print(f'This was how many Mini-Neptunes matched Modern Earth: {len(modern_like_mini_neptunes.keys())}')

This was how many Mini-Neptunes matched Archean Earth: 5622
This was how many Mini-Neptunes matched Modern Earth: 175


### Then I went ahead and looked into the results a little more:

#### Modern Earth-like Mini-Neptune Space & saved as numpy space

In [11]:
modern_input_cases = []
for inputs in list(modern_like_mini_neptunes.keys()):
    
    rad = rad_plan_earth_units[inputs[0]]
    metal = log10_planet_metallicity[inputs[1]]
    tint = tint_K[inputs[2]]
    semi_major = semi_major_AU[inputs[3]]
    ctoO = ctoO_solar[inputs[4]]
    kzz = kzz_list[inputs[5]]
    
    modern_input_cases.append([rad, metal, tint, semi_major, ctoO, kzz])

matrix_modern_input_cases = np.array(modern_input_cases)
phases = np.array([0])

cases = np.array([[*row, phi] for row in matrix_modern_input_cases for phi in phases])

np.save('data/grid_results/modernearth_175minimatches_phase0.npy', cases)

print(cases.shape)

(175, 7)


#### Archean Earth-like Mini-Neptune Space & saved as numpy space

In [12]:
archean_input_cases = []
for inputs in list(archean_like_mini_neptunes.keys()):
    
    rad = rad_plan_earth_units[inputs[0]]
    metal = log10_planet_metallicity[inputs[1]]
    tint = tint_K[inputs[2]]
    semi_major = semi_major_AU[inputs[3]]
    ctoO = ctoO_solar[inputs[4]]
    kzz = kzz_list[inputs[5]]
    
    archean_input_cases.append([rad, metal, tint, semi_major, ctoO, kzz])

matrix_archean_input_cases = np.array(archean_input_cases)
phases = np.array([0])

cases = np.array([[*row, phi] for row in matrix_archean_input_cases for phi in phases])

np.save('data/grid_results/archeanearth_5622minimatches_phase0.npy', cases)

print(cases.shape)

(5622, 7)
